<a href="https://colab.research.google.com/github/20233l001039-coder/Curso_Llama/blob/main/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [4]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [5]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [6]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)

**Paso a paso**

1. **Velocidades de cada tren**  
   - Tren 1 (el primero): \(v_1 = 80 \text{ km/h}\)  
   - Tren 2 (el segundo): \(v_2 = 120 \text{ km/h}\)

2. **Tiempo de ventaja del tren 1**  
   - El tren 2 sale 2 h después.  
   - En esas 2 h el tren 1 recorre  
     \[
     d_{\text{adelanto}} = v_1 \times 2\ \text{h}= 80 \text{ km/h}\times 2\text{ h}=160 \text{ km}.
     \]
   - Así que cuando el tren 2 comienza, el tren 1 ya está a 160 km delante.

3. **Velocidad relativa**  
   - Ambos trenes viajan en la misma dirección, por lo que la velocidad con la que el tren 2 se acerca al tren 1 es  
     \[
     v_{\text{rel}} = v_2 - v_1 = 120 \text{ km/h} - 80 \text{ km/h}=40 \text{ km/h}.
     \]

4. **Tiempo de captura**  
   - Usando la fórmula \( \text{tiempo} = \frac{\text{distancia}}{\text{velocidad}} \):
     \[
     t_{\text{captura}} = \frac{d_{\text{adelanto}}}{v_{\text{rel}}} = \frac{160 \text{ km}}{40 \text{ km/h}} = 4 \text{ h}.
     \]

5. **Resultado**  
   - El tren 

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [8]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Ganó el equipo “CodeForce” con su app de IA.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [9]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [10]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [11]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [12]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

No. Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [13]:
# Leer API key, instalar e importar librerías
!pip install groq sentence-transformers --quiet

from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import numpy as np

# Inicializar cliente de Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print("Cliente de Groq listo.")

Cliente de Groq listo.


In [14]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "La calificación final del curso se compone de: 40% proyecto final, 30% retos semanales, "
    "20% participación en clase y 10% quiz de diagnóstico.",

    "Para aprobar el curso se requiere una calificación mínima de 70/100. Si el alumno obtiene "
    "entre 60 y 69, tiene derecho a un examen de recuperación único.",

    "Las entregas fuera de plazo tienen una penalización del 20% por cada día de retraso. "
    "Después de 3 días, la entrega no se acepta y se califica como 0."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [15]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])

    # Similitud coseno
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    norma_docs = np.linalg.norm(embeddings_documentos, axis=1)
    norma_preg = np.linalg.norm(embedding_pregunta)
    similitudes = similitudes / (norma_docs * norma_preg + 1e-8)

    indice_mas_similar = int(np.argmax(similitudes))
    return documentos[indice_mas_similar]

# Prueba rápida
pregunta_prueba = "¿Qué pasa si entrego mi tarea tarde?"
print("Fragmento recuperado:", buscar_fragmento(pregunta_prueba))

Fragmento recuperado: Las entregas fuera de plazo tienen una penalización del 20% por cada día de retraso. Después de 3 días, la entrega no se acepta y se califica como 0.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [16]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
pregunta = "¿Cuánto porcentaje de mi calificación final vale el proyecto final?"

respuesta_sin_rag_obj = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta}]
)

respuesta_sin_rag = respuesta_sin_rag_obj.choices[0].message.content
print("Sin RAG:", respuesta_sin_rag)

Sin RAG: El porcentaje que el proyecto final aporta a tu calificación final varía según la política de la asignatura (o el profesor).  

**En líneas generales:**

| Tipo de curso | Peso típico del proyecto final |
|---------------|-------------------------------|
| Cursos de grado (máster, licenciatura) | 20 – 30 % del total |
| Cursos de grado superior o formación profesional | 15 – 25 % del total |
| Cursos de universidad con enfoque práctico | 25 – 35 % del total |
| Cursos con evaluación continua (exámenes, prácticas, trabajos) | 10 – 20 % del total |

**Para saber con exactitud tu caso**:

1. **Revisa el plan de estudios o el syllabus** que te entregó el profesor al inicio del curso; allí suele quedar claro el porcentaje de cada elemento.
2. **Pregunta directamente al profesor o a la secretaría** si el documento no lo indica de forma explícita.
3. **Consulta la rúbrica de evaluación** que se suele publicar para los proyectos finales; a menudo muestra el valor porcentual.

Si me da

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [17]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)
print()

prompt_rag = f"""Responde la pregunta del alumno usando SOLO la siguiente información del reglamento. Si la información no cubre la pregunta, dilo claramente.

Reglamento: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_con_rag_obj = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

respuesta_con_rag = respuesta_con_rag_obj.choices[0].message.content
print("Con RAG:", respuesta_con_rag)

Fragmento recuperado: La calificación final del curso se compone de: 40% proyecto final, 30% retos semanales, 20% participación en clase y 10% quiz de diagnóstico.

Con RAG: 40%


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [18]:
# Mostrar ambas respuestas para comparar
print("=" * 60)
print("PREGUNTA:", pregunta)
print("=" * 60)
print("\n--- RESPUESTA SIN RAG ---")
print(respuesta_sin_rag)
print("\n--- RESPUESTA CON RAG ---")
print(respuesta_con_rag)
print("\n" + "=" * 60)

print("=== CONCLUSIÓN ===")
print("La respuesta CON RAG es más precisa porque se basa en el fragmento real del reglamento,")
print("mientras que la respuesta SIN RAG puede inventar porcentajes o mezclar información de")
print("otros contextos (alucinación). El RAG ancla al modelo a la fuente de verdad.")

PREGUNTA: ¿Cuánto porcentaje de mi calificación final vale el proyecto final?

--- RESPUESTA SIN RAG ---
El porcentaje que el proyecto final aporta a tu calificación final varía según la política de la asignatura (o el profesor).  

**En líneas generales:**

| Tipo de curso | Peso típico del proyecto final |
|---------------|-------------------------------|
| Cursos de grado (máster, licenciatura) | 20 – 30 % del total |
| Cursos de grado superior o formación profesional | 15 – 25 % del total |
| Cursos de universidad con enfoque práctico | 25 – 35 % del total |
| Cursos con evaluación continua (exámenes, prácticas, trabajos) | 10 – 20 % del total |

**Para saber con exactitud tu caso**:

1. **Revisa el plan de estudios o el syllabus** que te entregó el profesor al inicio del curso; allí suele quedar claro el porcentaje de cada elemento.
2. **Pregunta directamente al profesor o a la secretaría** si el documento no lo indica de forma explícita.
3. **Consulta la rúbrica de evaluación** q